In [25]:
from sklearn.model_selection import train_test_split
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

df = pd.read_csv('../data/job_salary_prediction_dataset.csv')
df.head()

x = df.drop('salary',axis=1)
y = df['salary']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

num = x.select_dtypes(include=['int64','float64']).columns.tolist()
cat = x.select_dtypes(include=['object']).columns.tolist()

print(num)
print(cat)

num_pipe = Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe = Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('one',OneHotEncoder(handle_unknown='ignore',drop='first'))])

preprocess = ColumnTransformer([('num',num_pipe,num),('cat',cat_pipe,cat)])

model = Pipeline([('preprocess',preprocess),('model',XGBRegressor(learning_rate=0.1,max_depth=6,n_estimators=200))])

model.fit(x_train,y_train)
pred = model.predict(x_test)
print(r2_score(pred,y_test))

print(df.columns)

/tmp/ipykernel_6902/3600880392.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = x.select_dtypes(include=['object']).columns.tolist()


['experience_years', 'skills_count', 'certifications']
['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']
0.9725756049156189
Index(['job_title', 'experience_years', 'education_level', 'skills_count',
       'industry', 'company_size', 'location', 'remote_work', 'certifications',
       'salary'],
      dtype='str')


In [7]:
import joblib
from sklearn.metrics import r2_score

model = joblib.load("model.pkl")
x_test,y_test = joblib.load('set.pkl')

prediction = model.predict(x_test)
acc = r2_score(prediction,y_test)

print('metrics',acc)

if acc <= 0.90:
    print("model was better at performing")
else:
    print("model was worst condition")

FileNotFoundError: [Errno 2] No such file or directory: 'model.pkl'

In [24]:
import os
import tensorflow as tf
from tensorflow.keras import layers 
from tensorflow.keras import Sequential

x_preprocess = preprocess.fit_transform(x)

model = Sequential([
layers.Dense(128, activation='relu'),
layers.Dense(64, activation='relu'),
layers.Dense(32, activation='relu'),
layers.Dense(1)
])

model.compile(optimizer='adam',loss='mse',metrics=[tf.keras.metrics.R2Score(name='r2')])
model.fit(x_preprocess,y,epochs=3)

model_dir = 'saved_models/my_models/1'
os.makedirs(model_dir,exist_ok=True)

tf.saved_model(model,model_dir)

Epoch 1/3
7813/7813 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - loss: 1268965760.0000 - r2: 0.0932
Epoch 2/3
7813/7813 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 31771016.0000 - r2: 0.9773
Epoch 3/3
7813/7813 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 28784464.0000 - r2: 0.9794


TypeError: 'module' object is not callable

In [6]:
import requests
import json
import pandas as pd
import joblib

preprocess = joblib.load("../src/preprocess.pkl")

URL = "http://localhost:8501/v1/models/my_models:predict"

raw_data = pd.DataFrame([{
    "job_title": "AI Engineer",
    "experience_years": 10,
    "education_level": "Bachelor",
    "skills_count": 2,
    "industry": "Healthcare",
    "company_size": "Medium",
    "location": "India",
    "remote_work": "Hybrid",
    "certifications": 2
}])

processed = preprocess.transform(raw_data)

payload = {
    "instances": processed.toarray().tolist()
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(
    URL,
    data=json.dumps(payload),
    headers=headers
)

print(response.text)
print(response.status_code)

{
    "predictions": [[104588.195]
    ]
}
200


In [1]:
import mlflow.sklearn
import optuna
import mlflow
from sklearn.metrics import f1_score,recall_score,accuracy_score 
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,classification_report, f1_score
import pandas as pd

df = pd.read_csv("../data/customer_churn_dataset-testing-master.csv")

print(df.head())
print(df["Churn"].value_counts())

x = df.drop(["Churn","CustomerID"],axis=1)

le = LabelEncoder()
y = le.fit_transform(df["Churn"])

mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('churn_dataset')


num = x.select_dtypes(include=['int64','float64']).columns.tolist()
cat = x.select_dtypes(include=['object']).columns.tolist()
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

num_pipe = Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())])

cat_pipe = Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("one",OneHotEncoder(handle_unknown="ignore",drop="first"))])

preprocess = ColumnTransformer([("num",num_pipe,num),("cat",cat_pipe,cat)])
def objective(trials):
    with mlflow.start_run(nested=True):
        params = {
            "learning_rate":trials.suggest_float('learning_rate',0.1,0.5),
            "n_estimators":trials.suggest_int('n_estimators',100,300),
            "max_depth":trials.suggest_int("max_depth",6,8),
        }

        model = Pipeline([("preprocess",preprocess),("model",XGBClassifier(**params))])
        mlflow.log_params(params)
        model.fit(x_train,y_train)
        pred = model.predict(x_test)
        acc = accuracy_score(y_test,pred)
        f1 = f1_score(y_test,pred)
        recall = recall_score(y_test,pred)
        mlflow.log_metric('accuracy',acc)
        mlflow.log_metric('f1',f1)  
        mlflow.log_metric('recall',recall)
        mlflow.log_param('dataset',"churn")
        
        return acc

with mlflow.start_run(run_name="optuna_search"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective,n_trials=10)


    best_model = Pipeline([('preprocess',preprocess),('model',XGBClassifier(**study.best_params))])
    best_model.fit(x_train,y_train)
    pred = best_model.predict(x_test)
    best_accuracy = accuracy_score(y_test,pred)
    best_f1 = f1_score(y_test,pred)
    best_recall = recall_score(y_test,pred)

    mlflow.log_params(study.best_params)
    mlflow.log_metric('accuracy',best_accuracy)
    mlflow.log_metric('f1',best_f1)
    mlflow.log_metric("recall",best_recall)
    mlflow.sklearn.log_model(best_model,name='model',serialization_format='pickle')


/mnt/d/uv/practise/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   CustomerID  Age  Gender  Tenure  Usage Frequency  Support Calls  \
0           1   22  Female      25               14              4   
1           2   41  Female      28               28              7   
2           3   47    Male      27               10              2   
3           4   35    Male       9               12              5   
4           5   53  Female      58               24              9   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0             27             Basic         Monthly          598   
1             13          Standard         Monthly          584   
2             29           Premium          Annual          757   
3             17           Premium       Quarterly          232   
4              2          Standard          Annual          533   

   Last Interaction  Churn  
0                 9      1  
1                20      0  
2                21      0  
3                18      0  
4                18      0  
Ch

/tmp/ipykernel_1614/3516732892.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = x.select_dtypes(include=['object']).columns.tolist()
[I 2026-11-04 17:36:27,303] A new study created in memory with name: no-name-51b3117b-e57e-4441-b1a5-a30b4040a06c
[I 2026-11-04 17:36:28,821] Trial 0 finished with value: 1.0 and parameters: {'learning_rate': 0.3592093319416668, 'n_estimators': 248, 'max_depth': 8}. Best is trial 0 with value: 1.0.


🏃 View run adaptable-ray-605 at: http://127.0.0.1:5000/#/experiments/1/runs/cbea696ef6a340629f7a4fd0e18db24c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-11-04 17:36:29,737] Trial 1 finished with value: 0.9999223300970874 and parameters: {'learning_rate': 0.48160313896289597, 'n_estimators': 174, 'max_depth': 6}. Best is trial 0 with value: 1.0.


🏃 View run indecisive-lamb-405 at: http://127.0.0.1:5000/#/experiments/1/runs/72bc9e0ef391463a829a06b7568091b4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


... (14 outputs left out)

[I 2026-11-04 17:36:37,845] Trial 9 finished with value: 1.0 and parameters: {'learning_rate': 0.13968601446032836, 'n_estimators': 227, 'max_depth': 6}. Best is trial 0 with value: 1.0.


🏃 View run skillful-fowl-78 at: http://127.0.0.1:5000/#/experiments/1/runs/b2c2d3fb4130413fac286b5503e1292a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/11/04 17:36:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/04 17:37:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run optuna_search at: http://127.0.0.1:5000/#/experiments/1/runs/66efa7cd36dd43bdb760942cc63183e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
